# Agentick Benchmark Dashboard

Unified analysis of **PPO training** and **LLM/API evaluation** results.
Main metric: **Success Rate (SR)** on the 25 eval seeds per (task, difficulty).

In [ ]:
import json
import re
import warnings
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

PALETTE = [
    "#E69F00", "#56B4E9", "#009E73", "#F0E442",
    "#0072B2", "#D55E00", "#CC79A7", "#000000",
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
]
DIFFICULTIES = ["easy", "medium", "hard", "expert"]
DIFF_COLORS = {"easy": "#009E73", "medium": "#0072B2", "hard": "#E69F00", "expert": "#D55E00"}
CATEGORIES = [
    "navigation", "planning", "reasoning", "memory",
    "generalization", "multi_agent",
]
TASK_CATEGORIES = {
    "GoToGoal-v0": "navigation", "MazeNavigation-v0": "navigation",
    "ShortestPath-v0": "navigation", "DynamicObstacles-v0": "navigation",
    "CuriosityMaze-v0": "navigation", "RecursiveRooms-v0": "navigation",
    "TimingChallenge-v0": "navigation", "InstructionFollowing-v0": "navigation",
    "SokobanPush-v0": "planning", "KeyDoorPuzzle-v0": "planning",
    "BacktrackPuzzle-v0": "planning", "TileSorting-v0": "planning",
    "PackingPuzzle-v0": "planning", "PreciseNavigation-v0": "planning",
    "RecipeAssembly-v0": "planning", "ToolUse-v0": "planning",
    "ResourceManagement-v0": "planning",
    "CausalChain-v0": "reasoning", "SwitchCircuit-v0": "reasoning",
    "RuleInduction-v0": "reasoning", "LightsOut-v0": "reasoning",
    "GraphColoring-v0": "reasoning", "SymbolMatching-v0": "reasoning",
    "ProgramSynthesis-v0": "reasoning", "TaskInterference-v0": "reasoning",
    "DeceptiveReward-v0": "reasoning",
    "SequenceMemory-v0": "memory", "DelayedGratification-v0": "memory",
    "TreasureHunt-v0": "memory", "FogOfWarExploration-v0": "memory",
    "FewShotAdaptation-v0": "generalization", "DistributionShift-v0": "generalization",
    "NoisyObservation-v0": "generalization",
    "CooperativeTransport-v0": "multi_agent", "TagHunt-v0": "multi_agent",
    "ChaseEvade-v0": "multi_agent", "Herding-v0": "multi_agent",
    "EmergentStrategy-v0": "multi_agent",
}

print("Imports ready.")

---
## 1. Load All Results (PPO + LLM/API)

In [ ]:
def find_project_root():
    p = Path.cwd()
    for _ in range(10):
        if (p / "results").is_dir():
            return p
        p = p.parent
    raise FileNotFoundError("Cannot find results/ directory.")


ROOT = find_project_root()
RESULTS_DIR = ROOT / "results"
print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

# Known task names for stripping from agent labels (backward compat)
ALL_TASK_NAMES = set(TASK_CATEGORIES.keys())
TASK_STEMS = {t.removesuffix("-v0") for t in ALL_TASK_NAMES}


def _strip_task_suffix(name: str) -> str:
    """Strip task name suffix from agent label (e.g. 'foo-EmergentStrategy' -> 'foo')."""
    for stem in sorted(TASK_STEMS, key=len, reverse=True):
        if name.endswith(f"-{stem}"):
            return name[: -(len(stem) + 1)]
    return name


def _sr_curve_from_npz(eval_dir):
    """Compute SR curve from SB3 evaluations.npz."""
    npz_path = eval_dir / "evaluations.npz"
    if not npz_path.exists():
        return []
    data = np.load(str(npz_path))
    curve = []
    for i, ts in enumerate(data["timesteps"]):
        ep_returns = data["results"][i]
        curve.append({"timestep": int(ts), "success_rate": float(np.mean(ep_returns > 0))})
    return curve


# ============================================================
# Load PPO runs from per_task/ directories (robust to SLURM splitting)
# ============================================================
ppo_records = []
ppo_runs = {}  # agent_label -> metadata

ppo_base = RESULTS_DIR / "ppo_benchmarks"
if ppo_base.exists():
    for ts_file in sorted(ppo_base.rglob("training_summary.json")):
        run_path = ts_file.parent

        # Read metadata from training_summary.json
        with open(ts_file) as f:
            ts_data = json.load(f)
        reward_mode = ts_data.get("reward_mode", "unknown")
        render_mode = ts_data.get("render_mode", "rgb_array")
        render_short = "flat" if "flat" in render_mode else "iso"
        agent_label = f"PPO ({reward_mode}, {render_short})"

        # Use agent_label as the run key so multiple SLURM job
        # directories with the same variant get merged.
        if agent_label not in ppo_runs:
            ppo_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "ppo",
                "reward_mode": reward_mode, "render_mode": render_mode,
                "total_timesteps": ts_data.get("total_timesteps", 0),
                "run_dirs": [],
            }
        ppo_runs[agent_label]["run_dirs"].append(run_path)

        # Load records from per_task/ directories (each file has one
        # task-difficulty result), which is authoritative regardless of
        # whether training_summary.json was overwritten by another job.
        per_task_dir = run_path / "per_task"
        loaded_from_disk = False
        if per_task_dir.exists():
            for task_dir in sorted(per_task_dir.iterdir()):
                if not task_dir.is_dir():
                    continue
                task_name = task_dir.name
                for diff_dir in sorted(task_dir.iterdir()):
                    if not diff_dir.is_dir():
                        continue
                    metrics_file = diff_dir / "metrics.json"
                    if not metrics_file.exists():
                        continue
                    with open(metrics_file) as f:
                        m = json.load(f)
                    task = m.get("task", task_name)
                    diff = m.get("difficulty", diff_dir.name)
                    cat = m.get("category", TASK_CATEGORIES.get(task, "unknown"))

                    # Load SR curve
                    sr_curve = []
                    tc = m.get("training_curve", [])
                    if tc and isinstance(tc[0], dict) and "success_rate" in tc[0]:
                        sr_curve = [
                            {"timestep": p["timestep"], "success_rate": p["success_rate"]}
                            for p in tc
                        ]
                    else:
                        eval_dir = diff_dir / "eval"
                        sr_curve = _sr_curve_from_npz(eval_dir)

                    ppo_records.append({
                        "run": agent_label, "agent_label": agent_label,
                        "agent_type": "ppo",
                        "task": task, "task_short": task.replace("-v0", ""),
                        "difficulty": diff, "category": cat,
                        "success_rate": m.get("success_rate", 0.0),
                        "mean_return": m.get("mean_return", 0.0),
                        "sr_curve": sr_curve,
                    })
                    loaded_from_disk = True

        # Fallback: if per_task/ was empty, read from training_summary.json
        if not loaded_from_disk:
            for key, m in ts_data.get("results", {}).items():
                task = m["task"]
                diff = m["difficulty"]
                cat = m.get("category", TASK_CATEGORIES.get(task, "unknown"))
                sr_curve = []
                tc = m.get("training_curve", [])
                if tc and isinstance(tc[0], dict) and "success_rate" in tc[0]:
                    sr_curve = [
                        {"timestep": p["timestep"], "success_rate": p["success_rate"]}
                        for p in tc
                    ]
                ppo_records.append({
                    "run": agent_label, "agent_label": agent_label,
                    "agent_type": "ppo",
                    "task": task, "task_short": task.replace("-v0", ""),
                    "difficulty": diff, "category": cat,
                    "success_rate": m.get("success_rate", 0.0),
                    "mean_return": m.get("mean_return", 0.0),
                    "sr_curve": sr_curve,
                })

# Deduplicate PPO records (same task+diff from overlapping summaries)
seen_ppo = set()
deduped_ppo = []
for r in ppo_records:
    key = (r["run"], r["task"], r["difficulty"])
    if key not in seen_ppo:
        seen_ppo.add(key)
        deduped_ppo.append(r)
ppo_records = deduped_ppo

print(f"PPO: {len(ppo_runs)} agent(s), {len(ppo_records)} records")

# ============================================================
# Load LLM/API runs (from results/llm_benchmarks/*/per_task/)
# ============================================================
llm_records = []
llm_runs = {}  # agent_label -> metadata

llm_base = RESULTS_DIR / "llm_benchmarks"
if llm_base.exists():
    for summary_file in sorted(llm_base.rglob("summary.json")):
        run_path = summary_file.parent

        # Read config to get agent label
        cfg_path = run_path / "config.yaml"
        agent_label = run_path.name  # fallback
        if cfg_path.exists():
            cfg_text = cfg_path.read_text()
            m_name = re.search(r"^name:\s*(.+)$", cfg_text, re.MULTILINE)
            if m_name:
                agent_label = m_name.group(1).strip()

        # Strip task suffix from agent label (backward compat with
        # old per-task configs that appended -TaskName to the name)
        agent_label = _strip_task_suffix(agent_label)

        if agent_label not in llm_runs:
            llm_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "llm",
                "run_dirs": [],
            }
        llm_runs[agent_label]["run_dirs"].append(run_path)

        # Load per-task metrics
        per_task_dir = run_path / "per_task"
        if not per_task_dir.exists():
            continue
        for task_dir in sorted(per_task_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            metrics_file = task_dir / "metrics.json"
            if not metrics_file.exists():
                continue
            with open(metrics_file) as f:
                task_data = json.load(f)

            task_name = task_data.get("task_name", task_dir.name)
            cat = TASK_CATEGORIES.get(task_name, "unknown")

            for diff, diff_data in task_data.get("per_difficulty", {}).items():
                metrics = diff_data.get("metrics", {})
                sr = metrics.get("success_rate", 0.0)
                mr = metrics.get("mean_return", 0.0)

                llm_records.append({
                    "run": agent_label, "agent_label": agent_label,
                    "agent_type": "llm",
                    "task": task_name, "task_short": task_name.replace("-v0", ""),
                    "difficulty": diff, "category": cat,
                    "success_rate": sr, "mean_return": mr,
                    "sr_curve": [],
                })

# Deduplicate LLM records
seen_llm = set()
deduped_llm = []
for r in llm_records:
    key = (r["run"], r["task"], r["difficulty"])
    if key not in seen_llm:
        seen_llm.add(key)
        deduped_llm.append(r)
llm_records = deduped_llm

print(f"LLM: {len(llm_runs)} agent(s), {len(llm_records)} records")

# ============================================================
# Combine into unified record list
# ============================================================
all_records = ppo_records + llm_records
all_runs = {**ppo_runs, **llm_runs}

# Build label and color maps — keyed by agent_label now
agent_labels = {label: info["agent_label"] for label, info in all_runs.items()}
run_names = sorted(all_runs.keys())
agent_colors = {rn: PALETTE[i % len(PALETTE)] for i, rn in enumerate(run_names)}
cat_colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(CATEGORIES)}

tasks_all = sorted(set(r["task"] for r in all_records))

print(f"\nTotal: {len(all_runs)} agent(s), {len(all_records)} records, {len(tasks_all)} tasks")
for rn in run_names:
    n = sum(1 for r in all_records if r["run"] == rn)
    print(f"  {agent_labels[rn]:40s} [{all_runs[rn]['agent_type']:>3s}] {n:>4d} records")

---
## 2. Main Benchmark Results: Eval SR Comparison

Bar chart comparing all agents (PPO + LLM) on eval success rate, averaged over all task-difficulty pairs.

In [ ]:
if len(run_names) == 0:
    print("No results found.")
else:
    # Overall SR per agent
    agent_sr = {}
    for rn in run_names:
        srs = [r["success_rate"] for r in all_records if r["run"] == rn]
        agent_sr[rn] = (np.mean(srs), np.std(srs)) if srs else (0, 0)

    sorted_agents = sorted(agent_sr.keys(), key=lambda rn: agent_sr[rn][0], reverse=True)

    fig, ax = plt.subplots(figsize=(max(6, len(sorted_agents) * 1.2), 5))
    x = np.arange(len(sorted_agents))
    means = [agent_sr[rn][0] for rn in sorted_agents]
    stds = [agent_sr[rn][1] for rn in sorted_agents]
    colors = [agent_colors[rn] for rn in sorted_agents]
    labels = [agent_labels[rn] for rn in sorted_agents]

    bars = ax.bar(x, means, yerr=stds, color=colors, capsize=4, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Mean Eval Success Rate")
    ax.set_ylim(0, 1.05)
    ax.set_title("Main Benchmark: Eval Success Rate (all tasks, all difficulties)",
                 fontweight="bold", fontsize=12)

    # Annotate bars
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{m:.1%}", ha="center", va="bottom", fontsize=8, fontweight="bold")

    plt.tight_layout()
    plt.show()

---
## 3. Eval SR by Category (All Agents)

In [ ]:
if len(run_names) >= 1:
    active_cats = [c for c in CATEGORIES if any(r["category"] == c for r in all_records)]
    n_agents = len(run_names)
    bar_width = 0.8 / max(n_agents, 1)

    fig, ax = plt.subplots(figsize=(max(8, len(active_cats) * 1.5), 5))
    x = np.arange(len(active_cats))

    for ri, rn in enumerate(sorted_agents if 'sorted_agents' in dir() else run_names):
        recs = [r for r in all_records if r["run"] == rn]
        cat_sr = defaultdict(list)
        for r in recs:
            cat_sr[r["category"]].append(r["success_rate"])

        vals = [np.mean(cat_sr[c]) if c in cat_sr else 0 for c in active_cats]
        errs = [np.std(cat_sr[c]) if c in cat_sr else 0 for c in active_cats]
        offset = (ri - n_agents / 2 + 0.5) * bar_width
        ax.bar(x + offset, vals, bar_width, yerr=errs, label=agent_labels[rn],
               color=agent_colors[rn], capsize=3, edgecolor="white", linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("_", " ").title() for c in active_cats], rotation=30, ha="right")
    ax.set_ylabel("Mean Eval Success Rate")
    ax.set_ylim(0, 1.05)
    ax.set_title("Eval SR by Category", fontweight="bold")
    ax.legend(fontsize=6, loc="upper right")
    plt.tight_layout()
    plt.show()

---
## 4. Eval SR by Difficulty (All Agents)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for rn in run_names:
    recs = [r for r in all_records if r["run"] == rn]
    sr_by_diff = []
    sr_err = []
    for d in DIFFICULTIES:
        srs = [r["success_rate"] for r in recs if r["difficulty"] == d]
        sr_by_diff.append(np.mean(srs) if srs else 0)
        sr_err.append(np.std(srs) if srs else 0)

    ax.errorbar(DIFFICULTIES, sr_by_diff, yerr=sr_err, marker="o", linewidth=2,
                markersize=6, capsize=4, label=agent_labels[rn], color=agent_colors[rn])

ax.set_ylabel("Mean Eval Success Rate")
ax.set_xlabel("Difficulty")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Eval SR vs Difficulty", fontweight="bold")
ax.legend(fontsize=6)
plt.tight_layout()
plt.show()

---
## 5. Per-Task Eval SR Comparison (All Agents)

Side-by-side horizontal bars for each task, averaged over difficulties.

In [ ]:
if len(run_names) >= 2:
    task_agent_sr = defaultdict(dict)
    for r in all_records:
        task_agent_sr[r["task"]].setdefault(r["run"], []).append(r["success_rate"])

    # Sort tasks by mean SR across all agents
    task_mean = {t: np.mean([np.mean(v) for v in ag.values()])
                 for t, ag in task_agent_sr.items()}
    sorted_tasks = sorted(task_mean.keys(), key=lambda t: task_mean[t])

    fig, ax = plt.subplots(figsize=(10, max(6, len(sorted_tasks) * 0.35)))
    y = np.arange(len(sorted_tasks))
    n_agents = len(run_names)
    height = 0.8 / n_agents

    for ri, rn in enumerate(run_names):
        vals = [np.mean(task_agent_sr[t].get(rn, [0])) for t in sorted_tasks]
        offset = (ri - n_agents / 2 + 0.5) * height
        ax.barh(y + offset, vals, height, label=agent_labels[rn],
                color=agent_colors[rn], edgecolor="white", linewidth=0.5)

    ax.set_yticks(y)
    ax.set_yticklabels([t.replace("-v0", "") for t in sorted_tasks], fontsize=7)
    ax.set_xlabel("Mean Eval Success Rate")
    ax.set_xlim(0, 1.05)
    ax.set_title("Per-Task Eval SR Comparison (avg over difficulties)", fontweight="bold")
    ax.legend(loc="lower right", fontsize=6)
    plt.tight_layout()
    plt.show()
elif len(run_names) == 1:
    rn = run_names[0]
    recs = [r for r in all_records if r["run"] == rn]
    task_stats = defaultdict(lambda: {"sr": [], "cat": "unknown"})
    for r in recs:
        task_stats[r["task"]]["sr"].append(r["success_rate"])
        task_stats[r["task"]]["cat"] = r["category"]

    sorted_tasks = sorted(task_stats.keys(), key=lambda t: np.mean(task_stats[t]["sr"]))
    fig, ax = plt.subplots(figsize=(8, max(5, len(sorted_tasks) * 0.3)))
    names = [t.replace("-v0", "") for t in sorted_tasks]
    sr_means = [np.mean(task_stats[t]["sr"]) for t in sorted_tasks]
    colors = [cat_colors.get(task_stats[t]["cat"], "#999") for t in sorted_tasks]
    ax.barh(range(len(names)), sr_means, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel("Mean Eval Success Rate")
    ax.set_xlim(0, 1.05)
    ax.set_title(f"Per-Task SR — {agent_labels[rn]}", fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 6. SR Heatmap (Tasks x Difficulties) — Per Agent

In [ ]:
for rn in run_names:
    recs = [r for r in all_records if r["run"] == rn]

    # Order tasks by category
    cat_tasks = defaultdict(list)
    for r in recs:
        if r["task"] not in cat_tasks[r["category"]]:
            cat_tasks[r["category"]].append(r["task"])
    for c in cat_tasks:
        cat_tasks[c].sort()

    ordered_tasks = []
    cat_breaks = []
    cat_labels_list = []
    for cat in CATEGORIES:
        if cat in cat_tasks and cat_tasks[cat]:
            cat_breaks.append(len(ordered_tasks))
            cat_labels_list.append(cat)
            ordered_tasks.extend(cat_tasks[cat])

    if not ordered_tasks:
        continue

    lookup = {(r["task"], r["difficulty"]): r for r in recs}
    n_t, n_d = len(ordered_tasks), len(DIFFICULTIES)
    matrix = np.full((n_t, n_d), np.nan)
    for i, t in enumerate(ordered_tasks):
        for j, d in enumerate(DIFFICULTIES):
            if (t, d) in lookup:
                matrix[i, j] = lookup[(t, d)]["success_rate"]

    fig_h = max(8, n_t * 0.32)
    fig, ax = plt.subplots(figsize=(6, fig_h))
    im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest")

    ax.set_xticks(range(n_d))
    ax.set_xticklabels([d.capitalize() for d in DIFFICULTIES])
    ax.set_yticks(range(n_t))
    ax.set_yticklabels([t.replace("-v0", "") for t in ordered_tasks], fontsize=7)

    for i in range(n_t):
        for j in range(n_d):
            val = matrix[i, j]
            if not np.isnan(val):
                color = "white" if val < 0.35 or val > 0.85 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=6,
                        color=color, fontweight="bold")

    for brk in cat_breaks[1:]:
        ax.axhline(y=brk - 0.5, color="black", linewidth=1.2)

    for idx, (brk, label) in enumerate(zip(cat_breaks, cat_labels_list)):
        nxt = cat_breaks[idx + 1] if idx + 1 < len(cat_breaks) else n_t
        mid = (brk + nxt - 1) / 2
        ax.text(n_d + 0.2, mid, label.replace("_", " ").title(),
                va="center", fontsize=7, fontstyle="italic", color="#555")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.12)
    cbar.set_label("Success Rate")
    ax.set_title(f"Eval SR Heatmap — {agent_labels[rn]}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Difficulty")
    plt.tight_layout()
    plt.show()

---
## 7. Capability Radar (All Agents)

Radar chart showing per-category eval SR for each agent.

In [ ]:
active_cats = [c for c in CATEGORIES if any(r["category"] == c for r in all_records)]

if len(active_cats) >= 3:
    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(projection="polar"))
    angles = np.linspace(0, 2 * np.pi, len(active_cats), endpoint=False).tolist()
    angles_plot = angles + angles[:1]

    for rn in run_names:
        recs = [r for r in all_records if r["run"] == rn]
        cat_sr = defaultdict(list)
        for r in recs:
            cat_sr[r["category"]].append(r["success_rate"])
        values = [np.mean(cat_sr.get(c, [0])) for c in active_cats]
        values_plot = values + values[:1]
        ax.plot(angles_plot, values_plot, "o-", linewidth=2, label=agent_labels[rn],
                color=agent_colors[rn])
        ax.fill(angles_plot, values_plot, alpha=0.1, color=agent_colors[rn])

    ax.set_xticks(angles)
    ax.set_xticklabels([c.replace("_", " ").title() for c in active_cats], fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title("Capability Radar (Eval SR)", pad=20, fontsize=12, fontweight="bold")
    ax.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {len(active_cats)} categories — need at least 3 for radar chart.")

---
## 8. PPO Only: Training SR Curves (per task)

Shows how success rate on the eval seeds evolves during PPO training.  
Each subplot is one task, with 4 difficulty curves. LLM agents have no training curves.

In [ ]:
ppo_run_names = [rn for rn in run_names if all_runs[rn]["agent_type"] == "ppo"]

for rn in ppo_run_names:
    recs = [r for r in ppo_records if r["run"] == rn and r["sr_curve"]]
    tasks_with_curves = sorted(set(r["task"] for r in recs))

    n = len(tasks_with_curves)
    if n == 0:
        print(f"No SR curves for {agent_labels[rn]}")
        continue

    n_cols = 4
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False)

    for ti, task in enumerate(tasks_with_curves):
        ax = axes[ti // n_cols][ti % n_cols]
        task_recs = [r for r in recs if r["task"] == task]

        for r in task_recs:
            curve = r["sr_curve"]
            ts = [p["timestep"] for p in curve]
            srs = [p["success_rate"] for p in curve]
            color = DIFF_COLORS.get(r["difficulty"], "#999")
            ax.plot(ts, srs, label=r["difficulty"], color=color, linewidth=1.2)

        ax.set_title(task.replace("-v0", ""), fontsize=8, fontweight="bold")
        ax.set_ylim(-0.05, 1.05)
        ax.tick_params(labelsize=6)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
        if ti == 0:
            ax.legend(fontsize=5)

    for idx in range(n, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    fig.suptitle(f"PPO Eval SR During Training — {agent_labels[rn]}",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

---
## 9. PPO Only: Eval SR During Training — By Category

Medium difficulty, grouped by category. Shows learning progress per task.

In [ ]:
for rn in ppo_run_names:
    recs = [r for r in ppo_records if r["run"] == rn and r["difficulty"] == "medium" and r["sr_curve"]]
    if not recs:
        recs = [r for r in ppo_records if r["run"] == rn and r["difficulty"] == "easy" and r["sr_curve"]]

    cat_recs = defaultdict(list)
    for r in recs:
        cat_recs[r["category"]].append(r)

    active_cats = [c for c in CATEGORIES if c in cat_recs]
    if not active_cats:
        print(f"No SR curves for {agent_labels[rn]}")
        continue

    n_cols = min(3, len(active_cats))
    n_rows = (len(active_cats) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)

    for idx, cat in enumerate(active_cats):
        ax = axes[idx // n_cols][idx % n_cols]
        for ti, r in enumerate(sorted(cat_recs[cat], key=lambda x: x["task"])):
            curve = r["sr_curve"]
            ts = [p["timestep"] for p in curve]
            srs = [p["success_rate"] for p in curve]
            color = PALETTE[ti % len(PALETTE)]
            ax.plot(ts, srs, label=r["task_short"], color=color, linewidth=1.2)

        ax.set_title(cat.replace("_", " ").title(), fontsize=10, fontweight="bold")
        ax.set_xlabel("Timesteps")
        ax.set_ylabel("Eval SR")
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=5, loc="best", ncol=2)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

    for idx in range(len(active_cats), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    fig.suptitle(f"PPO Eval SR by Category (medium) — {agent_labels[rn]}",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

---
## 10. PPO Only: Aggregate Training SR Curve

Mean eval SR across all tasks at each training timestep, per difficulty.

In [ ]:
for rn in ppo_run_names:
    fig, ax = plt.subplots(figsize=(8, 4.5))

    for diff in DIFFICULTIES:
        recs = [r for r in ppo_records if r["run"] == rn and r["difficulty"] == diff and r["sr_curve"]]
        if not recs:
            continue

        # Collect all unique timesteps
        all_ts = sorted(set(p["timestep"] for r in recs for p in r["sr_curve"]))
        if not all_ts:
            continue

        # For each timestep, average SR across all tasks that have data at that timestep
        mean_srs = []
        for ts in all_ts:
            srs_at_ts = []
            for r in recs:
                # Find closest timestep
                pts = {p["timestep"]: p["success_rate"] for p in r["sr_curve"]}
                if ts in pts:
                    srs_at_ts.append(pts[ts])
            if srs_at_ts:
                mean_srs.append(np.mean(srs_at_ts))
            else:
                mean_srs.append(np.nan)

        color = DIFF_COLORS.get(diff, "#999")
        ax.plot(all_ts, mean_srs, label=diff, color=color, linewidth=2)

    ax.set_xlabel("Training Timesteps")
    ax.set_ylabel("Mean Eval SR (across tasks)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"Aggregate Eval SR During Training — {agent_labels[rn]}", fontweight="bold")
    ax.legend()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    plt.tight_layout()
    plt.show()

---
## 11. Category x Difficulty Heatmap — Per Agent

In [ ]:
for rn in run_names:
    recs = [r for r in all_records if r["run"] == rn]
    cats = [c for c in CATEGORIES if any(r["category"] == c for r in recs)]
    if not cats:
        continue

    matrix = np.full((len(cats), len(DIFFICULTIES)), np.nan)
    for ci, cat in enumerate(cats):
        for di, diff in enumerate(DIFFICULTIES):
            vals = [r["success_rate"] for r in recs
                    if r["category"] == cat and r["difficulty"] == diff]
            if vals:
                matrix[ci, di] = np.mean(vals)

    fig, ax = plt.subplots(figsize=(6, max(3, len(cats) * 0.6)))
    im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest")

    ax.set_xticks(range(len(DIFFICULTIES)))
    ax.set_xticklabels([d.capitalize() for d in DIFFICULTIES])
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels([c.replace("_", " ").title() for c in cats], fontsize=9)

    for i in range(len(cats)):
        for j in range(len(DIFFICULTIES)):
            val = matrix[i, j]
            if not np.isnan(val):
                color = "white" if val < 0.35 or val > 0.85 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=10,
                        color=color, fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Eval SR")
    ax.set_title(f"Category x Difficulty — {agent_labels[rn]}", fontweight="bold")
    ax.set_xlabel("Difficulty")
    plt.tight_layout()
    plt.show()

---
## 12. Top & Bottom Performers — Per Agent

In [ ]:
K = 10

for rn in run_names:
    recs = [r for r in all_records if r["run"] == rn]
    task_avg = defaultdict(lambda: {"sr": [], "cat": "unknown"})
    for r in recs:
        task_avg[r["task"]]["sr"].append(r["success_rate"])
        task_avg[r["task"]]["cat"] = r["category"]

    task_sr = {t: np.mean(v["sr"]) for t, v in task_avg.items()}
    sorted_by_sr = sorted(task_sr.items(), key=lambda x: x[1], reverse=True)

    top_k = sorted_by_sr[:K]
    bottom_k = sorted_by_sr[-K:]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Top K
    names_top = [t.replace("-v0", "") for t, _ in top_k]
    vals_top = [v for _, v in top_k]
    cols_top = [cat_colors.get(task_avg[t]["cat"], "#999") for t, _ in top_k]
    ax1.barh(range(len(names_top)), vals_top, color=cols_top, edgecolor="white", linewidth=0.5)
    ax1.set_yticks(range(len(names_top)))
    ax1.set_yticklabels(names_top, fontsize=8)
    ax1.set_xlim(0, 1.05)
    ax1.set_xlabel("Mean Eval SR")
    ax1.set_title(f"Top {K}", fontweight="bold", color="#009E73")
    ax1.invert_yaxis()

    # Bottom K
    names_bot = [t.replace("-v0", "") for t, _ in bottom_k]
    vals_bot = [v for _, v in bottom_k]
    cols_bot = [cat_colors.get(task_avg[t]["cat"], "#999") for t, _ in bottom_k]
    ax2.barh(range(len(names_bot)), vals_bot, color=cols_bot, edgecolor="white", linewidth=0.5)
    ax2.set_yticks(range(len(names_bot)))
    ax2.set_yticklabels(names_bot, fontsize=8)
    ax2.set_xlim(0, 1.05)
    ax2.set_xlabel("Mean Eval SR")
    ax2.set_title(f"Bottom {K}", fontweight="bold", color="#D55E00")
    ax2.invert_yaxis()

    # Category legend
    used_cats = sorted(set(task_avg[t]["cat"] for t in task_avg))
    patches = [Patch(facecolor=cat_colors.get(c, "#999"), label=c.replace("_", " ").title())
               for c in used_cats]
    fig.legend(handles=patches, fontsize=6, loc="lower center", ncol=len(patches),
              bbox_to_anchor=(0.5, -0.05))
    fig.suptitle(f"Top & Bottom Performers — {agent_labels[rn]}", fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 13. Aggregate Stats Summary

In [ ]:
# Summary table
header = f"{'Agent':<45} {'Type':>4} {'Records':>8} {'Mean SR':>9} {'Best SR':>9}"
print(header)
print("-" * len(header))

for rn in run_names:
    recs = [r for r in all_records if r["run"] == rn]
    all_sr = [r["success_rate"] for r in recs]
    atype = all_runs[rn]["agent_type"]
    mean_sr = np.mean(all_sr) if all_sr else 0
    best_sr = max(all_sr) if all_sr else 0
    print(f"{agent_labels[rn]:<45} {atype:>4} {len(recs):>8} {mean_sr:>9.1%} {best_sr:>9.1%}")

# Distribution plot
if len(run_names) <= 4:
    fig, axes = plt.subplots(1, len(run_names), figsize=(5 * len(run_names), 4), squeeze=False)
    for i, rn in enumerate(run_names):
        ax = axes[0][i]
        srs = [r["success_rate"] for r in all_records if r["run"] == rn]
        ax.hist(srs, bins=20, color=agent_colors[rn], edgecolor="white", linewidth=0.5, range=(0, 1))
        ax.axvline(x=np.mean(srs), color="red", linestyle="--",
                   label=f"Mean: {np.mean(srs):.1%}")
        ax.set_xlabel("Eval SR")
        ax.set_ylabel("Count")
        ax.set_title(agent_labels[rn], fontsize=9, fontweight="bold")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

---
## 14. Metadata

In [ ]:
for rn in run_names:
    info = all_runs[rn]
    n_recs = sum(1 for r in all_records if r["run"] == rn)
    print(f"{'=' * 60}")
    print(f"Agent:  {info['agent_label']}")
    print(f"Type:   {info['agent_type']}")
    run_dirs = info.get("run_dirs", [info["path"]])
    print(f"Dirs:   {len(run_dirs)} result folder(s)")
    for d in run_dirs[:3]:
        print(f"        {d}")
    if len(run_dirs) > 3:
        print(f"        ... and {len(run_dirs) - 3} more")
    print(f"Records: {n_recs}")

    # Load metadata from first available dir
    for rd in run_dirs:
        meta_path = rd / "metadata.json"
        if meta_path.exists():
            with open(meta_path) as f:
                meta = json.load(f)
            print(f"  Timestamp:  {meta.get('timestamp', 'N/A')}")
            print(f"  CUDA:       {meta.get('cuda_device', 'N/A')}")
            git = meta.get('git_hash', 'N/A')
            print(f"  Git:        {git[:12] if git else 'N/A'}")
            break

    # PPO-specific metadata
    if info["agent_type"] == "ppo":
        print(f"  Reward:     {info.get('reward_mode', 'N/A')}")
        print(f"  Render:     {info.get('render_mode', 'N/A')}")
        ts = info.get('total_timesteps', 0)
        print(f"  Timesteps:  {ts:,}" if isinstance(ts, int) else f"  Timesteps:  {ts}")

    print()